In [15]:
import pandas as pd
import numpy as np
from pathlib import Path
import sqlite3

# Paths
BASE = Path("C:/Users/nanig/OneDrive/Desktop/bluestock_mf_capstone")
RAW = BASE / "data/raw/Bluestock_MF_Datasets"
PROCESSED = BASE / "data/processed"
PROCESSED.mkdir(exist_ok=True)

DB_PATH = BASE / "data/db/bluestock_mf.db"
DB_PATH.parent.mkdir(exist_ok=True)

print("✅ Libraries loaded!")
print(f"RAW: {RAW}")
print(f"PROCESSED: {PROCESSED}")

✅ Libraries loaded!
RAW: C:\Users\nanig\OneDrive\Desktop\bluestock_mf_capstone\data\raw\Bluestock_MF_Datasets
PROCESSED: C:\Users\nanig\OneDrive\Desktop\bluestock_mf_capstone\data\processed


In [18]:
# === CLEAN NAV HISTORY ===
nav = pd.read_csv(RAW / "02_nav_history.csv")
print(f"Original: {nav.shape}")
print("Columns:", nav.columns.tolist())  # Check exact column names!

# Clean dates
nav['date'] = pd.to_datetime(nav['date'], errors='coerce')

# Sort by amfi_code and date (USE 'amfi_code' NOT 'scheme_code')
nav = nav.sort_values(['amfi_code', 'date'])

# Remove duplicates
nav = nav.drop_duplicates(subset=['amfi_code', 'date'], keep='first')

# Validate NAV > 0
nav = nav[nav['nav'] > 0]

# Forward fill missing dates (weekends/holidays)
nav = nav.set_index('date')
nav = nav.groupby('amfi_code')['nav'].apply(lambda x: x.ffill())
nav = nav.reset_index()

# Convert nav to numeric
nav['nav'] = pd.to_numeric(nav['nav'], errors='coerce')
nav = nav.dropna(subset=['nav'])

# Save
nav.to_csv(PROCESSED / "clean_nav.csv", index=False)
print(f"✅ Cleaned NAV: {nav.shape}")
print(nav.head())

Original: (46000, 3)
Columns: ['amfi_code', 'date', 'nav']
✅ Cleaned NAV: (46000, 3)
   amfi_code       date       nav
0     100016 2022-01-03  520.4608
1     100016 2022-01-04  515.0971
2     100016 2022-01-05  521.7239
3     100016 2022-01-06  515.7880
4     100016 2022-01-07  515.1639


In [19]:
# === CLEAN INVESTOR TRANSACTIONS ===
tx = pd.read_csv(RAW / "08_investor_transactions.csv")
print(f"Original: {tx.shape}")
print("Columns:", tx.columns.tolist())

# Clean dates
tx['transaction_date'] = pd.to_datetime(tx['transaction_date'], errors='coerce')

# Standardize transaction types
tx['transaction_type'] = tx['transaction_type'].str.strip().str.title()

# Validate amount > 0
tx = tx[tx['amount_inr'] > 0]

# Validate KYC status
valid_kyc = ['Verified', 'Pending', 'Rejected']
tx = tx[tx['kyc_status'].isin(valid_kyc)]

# Remove duplicates
tx = tx.drop_duplicates()

# Save
tx.to_csv(PROCESSED / "clean_transactions.csv", index=False)
print(f"✅ Cleaned Transactions: {tx.shape}")
print(tx.head())

Original: (32778, 13)
Columns: ['investor_id', 'transaction_date', 'amfi_code', 'transaction_type', 'amount_inr', 'state', 'city', 'city_tier', 'age_group', 'gender', 'annual_income_lakh', 'payment_mode', 'kyc_status']
✅ Cleaned Transactions: (32778, 13)
  investor_id transaction_date  amfi_code transaction_type  amount_inr  \
0   INV003054       2024-01-01     119092              Sip        1834   
1   INV002952       2024-01-01     148567       Redemption      392882   
2   INV003420       2024-01-01     118636              Sip         912   
3   INV003436       2024-01-01     118634              Sip        1102   
4   INV004691       2024-01-01     119094          Lumpsum        8682   

         state       city city_tier age_group  gender  annual_income_lakh  \
0    Telangana  Hyderabad       T30       56+  Female                77.1   
1       Punjab   Amritsar       B30     18-25    Male                 7.1   
2      Haryana  Faridabad       B30     36-45    Male                

In [20]:
# === CLEAN SCHEME PERFORMANCE ===
perf = pd.read_csv(RAW / "07_scheme_performance.csv")
print(f"Original: {perf.shape}")
print("Columns:", perf.columns.tolist())

# Validate returns are numeric
perf['return_1yr_pct'] = pd.to_numeric(perf['return_1yr_pct'], errors='coerce')
perf['return_3yr_pct'] = pd.to_numeric(perf['return_3yr_pct'], errors='coerce')
perf['return_5yr_pct'] = pd.to_numeric(perf['return_5yr_pct'], errors='coerce')

# Drop rows with missing returns
perf = perf.dropna(subset=['return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct'])

# Validate expense ratio range (0.1% - 2.5%)
perf = perf[(perf['expense_ratio_pct'] >= 0.1) & (perf['expense_ratio_pct'] <= 2.5)]

# Save
perf.to_csv(PROCESSED / "clean_performance.csv", index=False)
print(f"✅ Cleaned Performance: {perf.shape}")
print(perf.head())

Original: (40, 19)
Columns: ['amfi_code', 'scheme_name', 'fund_house', 'category', 'plan', 'return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct', 'benchmark_3yr_pct', 'alpha', 'beta', 'sharpe_ratio', 'sortino_ratio', 'std_dev_ann_pct', 'max_drawdown_pct', 'aum_crore', 'expense_ratio_pct', 'morningstar_rating', 'risk_grade']
✅ Cleaned Performance: (40, 19)
   amfi_code                                   scheme_name       fund_house  \
0     119551     SBI Bluechip Fund - Regular Plan - Growth  SBI Mutual Fund   
1     119552      SBI Bluechip Fund - Direct Plan - Growth  SBI Mutual Fund   
2     119598    SBI Small Cap Fund - Regular Plan - Growth  SBI Mutual Fund   
3     119599     SBI Small Cap Fund - Direct Plan - Growth  SBI Mutual Fund   
4     119120  SBI Magnum Gilt Fund - Regular Plan - Growth  SBI Mutual Fund   

    category     plan  return_1yr_pct  return_3yr_pct  return_5yr_pct  \
0  Large Cap  Regular           12.42           12.36           14.45   
1  Large Cap   Direct 

In [23]:
# === CLEAN ALL REMAINING CSVs ===

# 1. Fund Master
fm = pd.read_csv(RAW / "01_fund_master.csv")
fm.to_csv(PROCESSED / "clean_fund_master.csv", index=False)
print(f"✅ Fund Master: {fm.shape}")

# 2. AUM by Fund House
aum = pd.read_csv(RAW / "03_aum_by_fund_house.csv")
aum.to_csv(PROCESSED / "clean_aum.csv", index=False)
print(f"✅ AUM: {aum.shape}")

# 3. Monthly SIP Inflows
sip = pd.read_csv(RAW / "04_monthly_sip_inflows.csv")
sip.to_csv(PROCESSED / "clean_sip.csv", index=False)
print(f"✅ SIP Inflows: {sip.shape}")

# 4. Category Inflows
cat = pd.read_csv(RAW / "05_category_inflows.csv")
cat.to_csv(PROCESSED / "clean_cat.csv", index=False)
print(f"✅ Category Inflows: {cat.shape}")

# 5. Folio Count
folio = pd.read_csv(RAW / "06_industry_folio_count.csv")
folio.to_csv(PROCESSED / "clean_folio.csv", index=False)
print(f"✅ Folio Count: {folio.shape}")

# 6. Portfolio Holdings
port = pd.read_csv(RAW / "09_portfolio_holdings.csv")
port.to_csv(PROCESSED / "clean_portfolio.csv", index=False)
print(f"✅ Portfolio: {port.shape}")

# 7. Benchmark Indices
bench = pd.read_csv(RAW / "10_benchmark_indices.csv")
bench.to_csv(PROCESSED / "clean_benchmark.csv", index=False)
print(f"✅ Benchmark: {bench.shape}")

print("\n🎉 ALL DATASETS CLEANED!")

✅ Fund Master: (40, 15)
✅ AUM: (90, 5)
✅ SIP Inflows: (48, 6)
✅ Category Inflows: (144, 3)
✅ Folio Count: (21, 6)
✅ Portfolio: (322, 8)
✅ Benchmark: (8050, 3)

🎉 ALL DATASETS CLEANED!


In [24]:
# === CREATE SQLITE DATABASE ===
conn = sqlite3.connect(DB_PATH)

# Load all cleaned data into database
nav = pd.read_csv(PROCESSED / "clean_nav.csv")
nav.to_sql('fact_nav', conn, if_exists='replace', index=False)
print(f"✅ fact_nav: {len(nav)} rows")

tx = pd.read_csv(PROCESSED / "clean_transactions.csv")
tx.to_sql('fact_transactions', conn, if_exists='replace', index=False)
print(f"✅ fact_transactions: {len(tx)} rows")

perf = pd.read_csv(PROCESSED / "clean_performance.csv")
perf.to_sql('fact_performance', conn, if_exists='replace', index=False)
print(f"✅ fact_performance: {len(perf)} rows")

fm = pd.read_csv(PROCESSED / "clean_fund_master.csv")
fm.to_sql('dim_fund', conn, if_exists='replace', index=False)
print(f"✅ dim_fund: {len(fm)} rows")

aum = pd.read_csv(PROCESSED / "clean_aum.csv")
aum.to_sql('fact_aum', conn, if_exists='replace', index=False)
print(f"✅ fact_aum: {len(aum)} rows")

bench = pd.read_csv(PROCESSED / "clean_benchmark.csv")
bench.to_sql('fact_benchmark', conn, if_exists='replace', index=False)
print(f"✅ fact_benchmark: {len(bench)} rows")

conn.close()
print(f"\n🎉 Database created: {DB_PATH}")

✅ fact_nav: 46000 rows
✅ fact_transactions: 32778 rows
✅ fact_performance: 40 rows
✅ dim_fund: 40 rows
✅ fact_aum: 90 rows
✅ fact_benchmark: 8050 rows

🎉 Database created: C:\Users\nanig\OneDrive\Desktop\bluestock_mf_capstone\data\db\bluestock_mf.db


In [25]:
# === 10 SQL QUERIES ===
conn = sqlite3.connect(DB_PATH)

# Query 1: Top 5 funds by AUM
q1 = pd.read_sql("""
SELECT scheme_name, aum_crore 
FROM fact_performance 
ORDER BY aum_crore DESC 
LIMIT 5
""", conn)
print("1️⃣ Top 5 Funds by AUM:")
print(q1)

# Query 2: Average NAV per month
q2 = pd.read_sql("""
SELECT substr(date,1,7) as month, 
ROUND(AVG(nav),2) as avg_nav 
FROM fact_nav 
GROUP BY month 
ORDER BY month 
LIMIT 5
""", conn)
print("\n2️⃣ Average NAV per Month:")
print(q2)

# Query 3: Funds with expense ratio < 1%
q3 = pd.read_sql("""
SELECT scheme_name, expense_ratio_pct 
FROM fact_performance 
WHERE expense_ratio_pct < 1.0
ORDER BY expense_ratio_pct ASC
""", conn)
print("\n3️⃣ Funds with Expense Ratio < 1%:")
print(q3)

# Query 4: Transactions by state
q4 = pd.read_sql("""
SELECT state, COUNT(*) as total_tx, 
SUM(amount_inr) as total_amount 
FROM fact_transactions 
GROUP BY state 
ORDER BY total_amount DESC
""", conn)
print("\n4️⃣ Transactions by State:")
print(q4)

# Query 5: SIP transactions total
q5 = pd.read_sql("""
SELECT transaction_type, 
COUNT(*) as count, 
SUM(amount_inr) as total 
FROM fact_transactions 
GROUP BY transaction_type
""", conn)
print("\n5️⃣ Transactions by Type:")
print(q5)

# Query 6: Top 5 funds by 1yr return
q6 = pd.read_sql("""
SELECT scheme_name, return_1yr_pct 
FROM fact_performance 
ORDER BY return_1yr_pct DESC 
LIMIT 5
""", conn)
print("\n6️⃣ Top 5 Funds by 1yr Return:")
print(q6)

# Query 7: Average Sharpe ratio
q7 = pd.read_sql("""
SELECT ROUND(AVG(sharpe_ratio),2) as avg_sharpe 
FROM fact_performance
""", conn)
print("\n7️⃣ Average Sharpe Ratio:")
print(q7)

# Query 8: Funds with positive alpha
q8 = pd.read_sql("""
SELECT scheme_name, alpha 
FROM fact_performance 
WHERE alpha > 0 
ORDER BY alpha DESC 
LIMIT 5
""", conn)
print("\n8️⃣ Top Funds by Alpha:")
print(q8)

# Query 9: Monthly transaction volume
q9 = pd.read_sql("""
SELECT substr(transaction_date,1,7) as month, 
COUNT(*) as transactions 
FROM fact_transactions 
GROUP BY month 
ORDER BY month 
LIMIT 5
""", conn)
print("\n9️⃣ Monthly Transaction Volume:")
print(q9)

# Query 10: High risk funds
q10 = pd.read_sql("""
SELECT scheme_name, risk_grade 
FROM fact_performance 
WHERE risk_grade = 'Very High'
""", conn)
print("\n🔟 Very High Risk Funds:")
print(q10)

conn.close()
print("\n🎉 ALL QUERIES DONE!")

1️⃣ Top 5 Funds by AUM:
                                         scheme_name  aum_crore
0  Mirae Asset Emerging Bluechip Fund - Regular -...      49046
1      Kotak Emerging Equity Fund - Regular - Growth      47469
2     Nippon India Small Cap Fund - Regular - Growth      43630
3         DSP Top 100 Equity Fund - Regular - Growth      41828
4                UTI Mid Cap Fund - Regular - Growth      41728

2️⃣ Average NAV per Month:
     month  avg_nav
0  2022-01   207.06
1  2022-02   207.72
2  2022-03   209.69
3  2022-04   211.83
4  2022-05   212.73

3️⃣ Funds with Expense Ratio < 1%:
                                          scheme_name  expense_ratio_pct
0   Nippon India Gilt Securities Fund - Regular - ...               0.55
1        HDFC Short Term Debt Fund - Regular - Growth               0.56
2                Kotak Liquid Fund - Regular - Growth               0.60
3            SBI Bluechip Fund - Direct Plan - Growth               0.66
4           SBI Small Cap Fund - Direct Pla

In [26]:
import sqlite3

conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

schema = """
DROP TABLE IF EXISTS dim_fund;
DROP TABLE IF EXISTS fact_nav;
DROP TABLE IF EXISTS fact_transactions;
DROP TABLE IF EXISTS fact_performance;
DROP TABLE IF EXISTS fact_aum;
DROP TABLE IF EXISTS fact_benchmark;

CREATE TABLE dim_fund (
    amfi_code INTEGER PRIMARY KEY,
    scheme_name TEXT,
    fund_house TEXT,
    category TEXT,
    sub_category TEXT,
    risk_grade TEXT,
    expense_ratio REAL
);

CREATE TABLE fact_nav (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    amfi_code INTEGER,
    date TEXT,
    nav REAL,
    FOREIGN KEY (amfi_code) REFERENCES dim_fund(amfi_code)
);

CREATE TABLE fact_transactions (
    txn_id INTEGER PRIMARY KEY AUTOINCREMENT,
    investor_id INTEGER,
    amfi_code INTEGER,
    transaction_type TEXT,
    amount REAL,
    transaction_date TEXT,
    state TEXT,
    kyc_status TEXT,
    FOREIGN KEY (amfi_code) REFERENCES dim_fund(amfi_code)
);

CREATE TABLE fact_performance (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    amfi_code INTEGER,
    return_1yr REAL,
    return_3yr REAL,
    return_5yr REAL,
    sharpe_ratio REAL,
    expense_ratio REAL,
    FOREIGN KEY (amfi_code) REFERENCES dim_fund(amfi_code)
);

CREATE TABLE fact_aum (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    fund_house TEXT,
    date TEXT,
    aum_crore REAL
);

CREATE TABLE fact_benchmark (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    index_name TEXT,
    date TEXT,
    closing_price REAL
);
"""

cursor.executescript(schema)
conn.commit()
print("✅ Schema created!")

✅ Schema created!
